# Multi-Source Retail Sales Data Integration and Analysis



## 0. Setup — Install & Load Packages

In [1]:
install.packages(c("readr", "jsonlite", "readxl", "writexl", "dplyr",
                    "tidyr", "lubridate", "DBI", "RSQLite", "stringr"),
                  repos = "https://cloud.r-project.org")

suppressPackageStartupMessages({
  library(readr)      # CSV I/O
  library(jsonlite)   # JSON I/O
  library(readxl)     # Excel import
  library(writexl)    # Excel export
  library(dplyr)       # data wrangling
  library(tidyr)       # cleaning helpers
  library(lubridate)   # date parsing
  library(DBI)          # database interface
  library(RSQLite)      # SQLite backend
  library(stringr)      # string cleaning
})

options(scipen = 999)
set.seed(42)


Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



## 1. Fetch the Dataset Automatically


In [2]:
uci_primary <- "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
uci_mirror  <- "https://raw.githubusercontent.com/aungsan9418/AB_test_datasets/master/Online%20Retail.xlsx"

fetch_online_retail <- function() {
  local_path <- "Online_Retail_raw.xlsx"

  ok <- tryCatch({
    download.file(uci_primary, local_path, mode = "wb", quiet = TRUE, timeout = 120)
    TRUE
  }, error = function(e) FALSE)

  if (!ok || !file.exists(local_path) || file.size(local_path) < 1000) {
    message("Primary UCI link unavailable, trying mirror...")
    download.file(uci_mirror, local_path, mode = "wb", quiet = TRUE, timeout = 120)
  }

  read_excel(local_path)
}

raw_data <- fetch_online_retail()
cat("Downloaded raw dataset ->", nrow(raw_data), "rows,", ncol(raw_data), "columns\n")
head(raw_data)


Downloaded raw dataset -> 541909 rows, 8 columns


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
<chr>,<chr>,<chr>,<dbl>,<dttm>,<dbl>,<dbl>,<chr>
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom


In [3]:
transactions_src <- raw_data %>%
  select(InvoiceNo, StockCode, CustomerID, Quantity, InvoiceDate) %>%
  mutate(InvoiceDate = format(InvoiceDate, "%Y-%m-%d %H:%M:%S"))
write_csv(transactions_src, "transactions.csv")

products_src <- raw_data %>%
  distinct(StockCode, .keep_all = TRUE) %>%
  select(StockCode, Description, UnitPrice)
write_json(products_src, "products.json", pretty = TRUE, auto_unbox = TRUE)

customers_src <- raw_data %>%
  filter(!is.na(CustomerID)) %>%
  distinct(CustomerID, .keep_all = TRUE) %>%
  select(CustomerID, Country)
write_xlsx(customers_src, "customers.xlsx")

cat("transactions.csv :", nrow(transactions_src), "rows\n")
cat("products.json    :", nrow(products_src), "unique SKUs\n")
cat("customers.xlsx   :", nrow(customers_src), "unique customers\n")


transactions.csv : 541909 rows
products.json    : 4070 unique SKUs
customers.xlsx   : 4372 unique customers


## Task 1: Import and Clean the Data


In [4]:
transactions <- read_csv("transactions.csv", show_col_types = FALSE)
products     <- fromJSON("products.json")
customers    <- read_excel("customers.xlsx")

glimpse(transactions)
glimpse(products)
glimpse(customers)


Rows: 541,909
Columns: 5
$ InvoiceNo   <chr> "536365", "536365", "536365", "536365", "536365", "536365"…
$ StockCode   <chr> "85123A", "71053", "84406B", "84029G", "84029E", "22752", …
$ CustomerID  <dbl> 17850, 17850, 17850, 17850, 17850, 17850, 17850, 17850, 17…
$ Quantity    <dbl> 6, 6, 8, 6, 6, 2, 6, 6, 6, 32, 6, 6, 8, 6, 6, 3, 2, 3, 3, …
$ InvoiceDate <dttm> 2010-12-01 08:26:00, 2010-12-01 08:26:00, 2010-12-01 08:2…
Rows: 4,070
Columns: 3
$ StockCode   <chr> "85123A", "71053", "84406B", "84029G", "84029E", "22752", …
$ Description <chr> "WHITE HANGING HEART T-LIGHT HOLDER", "WHITE METAL LANTERN…
$ UnitPrice   <dbl> 2.55, 3.39, 2.75, 3.39, 3.39, 7.65, 4.25, 1.85, 1.85, 1.69…
Rows: 4,372
Columns: 2
$ CustomerID <dbl> 17850, 13047, 12583, 13748, 15100, 15291, 14688, 17809, 153…
$ Country    <chr> "United Kingdom", "United Kingdom", "France", "United Kingd…


### 1.1 Inspect Before Cleaning

In [5]:
cat("Dimensions -> transactions:", dim(transactions),
    "| products:", dim(products), "| customers:", dim(customers), "\n\n")

cat("Missing values (transactions):\n"); print(colSums(is.na(transactions)))
cat("\nMissing values (products):\n");     print(colSums(is.na(products)))
cat("\nMissing values (customers):\n");    print(colSums(is.na(customers)))

cat("\nDuplicate rows in transactions:", sum(duplicated(transactions)), "\n")
cat("Zero-quantity rows:", sum(transactions$Quantity == 0, na.rm = TRUE), "\n")
cat("Negative-quantity rows (returns/cancellations):", sum(transactions$Quantity < 0, na.rm = TRUE), "\n")
cat("Zero/negative-price products:", sum(products$UnitPrice <= 0, na.rm = TRUE), "\n")


Dimensions -> transactions: 541909 5 | products: 4070 3 | customers: 4372 2 

Missing values (transactions):
  InvoiceNo   StockCode  CustomerID    Quantity InvoiceDate 
          0           0      135080           0           0 

Missing values (products):
  StockCode Description   UnitPrice 
          0         176           0 

Missing values (customers):
CustomerID    Country 
         0          0 

Duplicate rows in transactions: 5429 
Zero-quantity rows: 0 
Negative-quantity rows (returns/cancellations): 10624 
Zero/negative-price products: 215 


### 1.2 Clean the Data


In [6]:
transactions_clean <- transactions %>%
  distinct() %>%
  filter(!is.na(CustomerID)) %>%
  filter(Quantity > 0) %>%
  mutate(InvoiceDate = as.POSIXct(InvoiceDate, format = "%Y-%m-%d %H:%M:%S"))

products_clean <- products %>%
  distinct(StockCode, .keep_all = TRUE) %>%
  filter(!is.na(UnitPrice), UnitPrice > 0, !is.na(Description))

customers_clean <- customers %>%
  distinct(CustomerID, .keep_all = TRUE) %>%
  filter(!is.na(Country))

cat("Transactions:", nrow(transactions), "->", nrow(transactions_clean), "after cleaning\n")
cat("Products    :", nrow(products), "->", nrow(products_clean), "after cleaning\n")
cat("Customers   :", nrow(customers), "->", nrow(customers_clean), "after cleaning\n")


Transactions: 541909 -> 392708 after cleaning
Products    : 4070 -> 3855 after cleaning
Customers   : 4372 -> 4372 after cleaning


### 1.3 Create the Revenue Attribute

In [7]:
transactions_clean <- transactions_clean %>%
  inner_join(products_clean %>% select(StockCode, UnitPrice), by = "StockCode") %>%
  mutate(Revenue = Quantity * UnitPrice)

cat("Rows retained after attaching a valid UnitPrice:", nrow(transactions_clean), "\n")
head(transactions_clean, 5)


Rows retained after attaching a valid UnitPrice: 387877 


InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,UnitPrice,Revenue
<chr>,<chr>,<dbl>,<dbl>,<dttm>,<dbl>,<dbl>
536365,85123A,17850,6,2010-12-01 08:26:00,2.55,15.30
536365,71053,17850,6,2010-12-01 08:26:00,3.39,20.34
536365,84406B,17850,8,2010-12-01 08:26:00,2.75,22.00
536365,84029G,17850,6,2010-12-01 08:26:00,3.39,20.34
536365,84029E,17850,6,2010-12-01 08:26:00,3.39,20.34


## Task 2: Integrate the Multiple Data Sources

In [8]:
retail_data <- transactions_clean %>%
  left_join(products_clean %>% select(StockCode, Description), by = "StockCode") %>%
  left_join(customers_clean, by = "CustomerID")

dim(retail_data)
head(retail_data)


[1] 387877      9

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,UnitPrice,Revenue,Description,Country
<chr>,<chr>,<dbl>,<dbl>,<dttm>,<dbl>,<dbl>,<chr>,<chr>
536365,85123A,17850,6,2010-12-01 08:26:00,2.55,15.30,WHITE HANGING HEART T-LIGHT HOLDER,United Kingdom
536365,71053,17850,6,2010-12-01 08:26:00,3.39,20.34,WHITE METAL LANTERN,United Kingdom
536365,84406B,17850,8,2010-12-01 08:26:00,2.75,22.00,CREAM CUPID HEARTS COAT HANGER,United Kingdom
536365,84029G,17850,6,2010-12-01 08:26:00,3.39,20.34,KNITTED UNION FLAG HOT WATER BOTTLE,United Kingdom
536365,84029E,17850,6,2010-12-01 08:26:00,3.39,20.34,RED WOOLLY HOTTIE WHITE HEART.,United Kingdom
536365,22752,17850,2,2010-12-01 08:26:00,7.65,15.30,SET 7 BABUSHKA NESTING BOXES,United Kingdom


### 2.1 Verify Dimensions & Unmatched Records

In [9]:
cat("transactions_clean rows:", nrow(transactions_clean), "\n")
cat("retail_data rows       :", nrow(retail_data), "\n\n")

unmatched_products  <- retail_data %>% filter(is.na(Description))
unmatched_customers <- retail_data %>% filter(is.na(Country))

cat("Rows with no matching product description:", nrow(unmatched_products), "\n")
cat("Rows with no matching customer/country    :", nrow(unmatched_customers), "\n")


transactions_clean rows: 387877 
retail_data rows       : 387877 

Rows with no matching product description: 0 
Rows with no matching customer/country    : 0 


### 2.2 Join Strategy — Justification

- **`inner_join()`** is used to attach `UnitPrice` from `products_clean`.
  A transaction without a valid, positive price cannot contribute to revenue,
  so it is correct to drop unmatched rows at this step.
- **`left_join()`** is used to attach `Description` and `Country`.
  Every transaction that already survived cleaning (and has a valid price)
  should be **kept** in the final table even if enrichment fields like the
  product description or customer country happen to be missing — dropping
  them here would silently lose real, revenue-generating sales.

## Task 3: Sales and Customer Analysis

### 3.1 Total Sales Revenue

In [10]:
total_revenue <- sum(retail_data$Revenue, na.rm = TRUE)
cat("Total Sales Revenue: £", format(round(total_revenue, 2), big.mark = ","), "\n")


Total Sales Revenue: £ 10,752,840 


### 3.2 Top 5 Products by Revenue

In [11]:
top5_products <- retail_data %>%
  group_by(StockCode, Description) %>%
  summarise(TotalRevenue = sum(Revenue, na.rm = TRUE), .groups = "drop") %>%
  arrange(desc(TotalRevenue)) %>%
  slice_head(n = 5)

top5_products


StockCode,Description,TotalRevenue
<chr>,<chr>,<dbl>
23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60
47566,PARTY BUNTING,142437.56
22423,REGENCY CAKESTAND 3 TIER,135604.80
85123A,WHITE HANGING HEART T-LIGHT HOLDER,93745.65
23166,MEDIUM CERAMIC TOP STORAGE JAR,81032.64


### 3.3 Top 5 Countries by Revenue

In [12]:
top5_countries <- retail_data %>%
  filter(!is.na(Country)) %>%
  group_by(Country) %>%
  summarise(TotalRevenue = sum(Revenue, na.rm = TRUE), .groups = "drop") %>%
  arrange(desc(TotalRevenue)) %>%
  slice_head(n = 5)

top5_countries


Country,TotalRevenue
<chr>,<dbl>
United Kingdom,8861857.1
Netherlands,363884.5
EIRE,331660.2
Germany,263819.0
France,226975.6


### 3.4 Top 5 Customers by Purchase Value

In [13]:
top5_customers <- retail_data %>%
  group_by(CustomerID) %>%
  summarise(TotalSpend = sum(Revenue, na.rm = TRUE), .groups = "drop") %>%
  arrange(desc(TotalSpend)) %>%
  slice_head(n = 5)

top5_customers


CustomerID,TotalSpend
<dbl>,<dbl>
18102,408760.0
14646,357531.1
17450,186038.0
14911,182690.5
16446,168472.5


### 3.5 Customer Value Segmentation

Customers are bucketed into four tiers using `case_when()`, with thresholds
based on the 25th/50th/75th percentile of total spend (a data-driven way to
pick "suitable thresholds" rather than arbitrary fixed numbers).

In [14]:
customer_value <- retail_data %>%
  group_by(CustomerID) %>%
  summarise(TotalSpend = sum(Revenue, na.rm = TRUE), .groups = "drop")

q <- quantile(customer_value$TotalSpend, probs = c(0.25, 0.50, 0.75), na.rm = TRUE)
print(q)

customer_value <- customer_value %>%
  mutate(ValueSegment = case_when(
    TotalSpend <= q[1] ~ "Low Value",
    TotalSpend <= q[2] ~ "Medium Value",
    TotalSpend <= q[3] ~ "High Value",
    TRUE               ~ "Premium"
  ))

table(customer_value$ValueSegment)


     25%      50%      75% 
 361.005  802.600 2000.640 



  High Value    Low Value Medium Value      Premium 
        1084         1085         1085         1085 

### 3.6 High-Performing vs Underperforming Market

In [15]:
country_revenue <- retail_data %>%
  filter(!is.na(Country)) %>%
  group_by(Country) %>%
  summarise(TotalRevenue = sum(Revenue, na.rm = TRUE),
            NumCustomers = n_distinct(CustomerID), .groups = "drop") %>%
  arrange(desc(TotalRevenue))

best_market  <- slice_head(country_revenue, n = 1)
worst_market <- slice_tail(country_revenue, n = 1)

cat("High-performing market:", best_market$Country,
    "| Revenue: £", format(round(best_market$TotalRevenue, 2), big.mark = ","),
    "| Customers:", best_market$NumCustomers, "\n")
cat("Underperforming market:", worst_market$Country,
    "| Revenue: £", round(worst_market$TotalRevenue, 2),
    "| Customers:", worst_market$NumCustomers, "\n")


High-performing market: United Kingdom | Revenue: £ 8,861,857 | Customers: 3921 
Underperforming market: Saudi Arabia | Revenue: £ 181.44 | Customers: 1 


## Task 4: Store and Retrieve Data Using SQL

In [16]:
con <- dbConnect(RSQLite::SQLite(), "retail_analysis.db")

dbWriteTable(con, "retail_sales", retail_data, overwrite = TRUE)

cat("Tables in DB:", dbListTables(con), "\n")
cat("Rows in retail_sales:",
    dbGetQuery(con, "SELECT COUNT(*) AS n FROM retail_sales")$n, "\n")


Tables in DB: retail_sales 
Rows in retail_sales: 387877 


### 4.1 SQL Query 1 — Top 5 Customers by Revenue

In [17]:
query1 <- dbGetQuery(con, "
  SELECT CustomerID, SUM(Revenue) AS TotalRevenue
  FROM retail_sales
  GROUP BY CustomerID
  ORDER BY TotalRevenue DESC
  LIMIT 5;
")
query1


CustomerID,TotalRevenue
<dbl>,<dbl>
18102,408760.0
14646,357531.1
17450,186038.0
14911,182690.5
16446,168472.5


### 4.2 SQL Query 2 — Total Revenue by Country

In [18]:
query2 <- dbGetQuery(con, "
  SELECT Country, SUM(Revenue) AS TotalRevenue
  FROM retail_sales
  WHERE Country IS NOT NULL
  GROUP BY Country
  ORDER BY TotalRevenue DESC;
")
head(query2, 10)

dbDisconnect(con)


,Country,TotalRevenue
,<chr>,<dbl>
1,United Kingdom,8861857.13
2,Netherlands,363884.48
3,EIRE,331660.17
4,Germany,263818.97
5,France,226975.60
6,Australia,173918.61
7,Spain,67426.09
8,Switzerland,66619.97
9,Japan,48600.22


## Conclusion: Business Insights



1. Revenue is heavily concentrated in the UK home market. The United Kingdom alone generated £8,861,857 out of £10,752,840 total revenue — about 82.4%, from 3,921 active customers — while the weakest market, Saudi Arabia, contributed just £181.44 from a single customer. This is the expected pattern for a UK-based online gift retailer, but it also means the business is heavily exposed to one geography, and countries like Netherlands (£363,884), EIRE (£331,660), and Germany (£263,819) represent the next tier worth targeted growth investment.
2. A small number of customers and products drive disproportionate revenue. The top 5 customers (led by CustomerID 18102 at £408,760 and 14646 at £357,531) contributed a combined ~£1.3M, and the top 5 SKUs — led by "PAPER CRAFT, LITTLE BIRDIE" (£168,470) and "PARTY BUNTING" (£142,438) — brought in over £621,000 between them, out of 4,372 customers and 3,855 valid products overall. The quartile-based case_when() segmentation splits customers almost evenly (1,084–1,085 per tier across Low/Medium/High/Premium), so a loyalty or account-management program aimed specifically at the "Premium" tier could protect and grow this concentrated value.
3. Upstream data quality has real gaps. 135,080 of 541,909 raw transactions (about 24.9%) had no CustomerID and had to be dropped, along with 5,429 duplicate rows, 10,624 negative-quantity (return/cancellation) rows, and 215 products with zero/negative prices — together shrinking the usable dataset from 541,909 to 387,877 clean, revenue-attributable rows (about 71.6% retained). This points to a clear operational fix: enforce mandatory CustomerID capture and price validation at the point of sale to reduce future data loss.